<a href="https://colab.research.google.com/github/2006-paneer/Convolve-4.0---A-Pan-IIT-AI-ML-Hackathon/blob/main/convolve_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv('train.csv')

In [ ]:
df.shape

(2017127, 31)

In [ ]:
df.dropna(inplace = True)

In [ ]:
df.shape

(2016825, 31)

In [ ]:
df1 = df.iloc[:,1:]

In [ ]:
df1.shape

(2016825, 30)

In [ ]:
X = df1.iloc[:,0:-1]
y = df1.iloc[:,-1]

In [ ]:
X.shape

(2016825, 29)

In [ ]:
y.shape

(2016825,)

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2,random_state = 42)

In [ ]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train_sc = sc.fit_transform(X_train)
X_test_sc = sc.transform(X_test)

In [ ]:
X_trainer,X_val,y_trainer,y_val = train_test_split(X_train_sc,y_train,test_size = 0.2,random_state = 42)

In [ ]:
X_train.shape

(1613460, 29)

In [ ]:
X_val.shape

(322692, 29)

In [ ]:
!pip install optuna
!pip install optuna_integration

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 5.6 MB/s eta 0:00:00


In [ ]:
import optuna
import optuna_integration
import xgboost as xgb

In [ ]:
from sklearn.metrics import root_mean_squared_error as rmse

In [ ]:
def objectivexgb(trial):
    # 2. Define the Search Space
    params = {
        'objective': 'reg:squarederror',
        'tree_method': 'hist',
        'device': 'cuda',  # Uses your GPU

        # Structure: Allow slightly deeper trees but constrain them heavily with min_child_weight
        'max_depth': trial.suggest_int('max_depth', 3, 7),

        # NOISE CONTROL: Higher values prevent the model from learning distinct noise patterns
        'min_child_weight': trial.suggest_int('min_child_weight', 20, 150),

        # Regularization: crucial for generalization
        'gamma': trial.suggest_float('gamma', 0.1, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 5.0),  # L1 reg
        'reg_lambda': trial.suggest_float('reg_lambda', 0.5, 5.0), # L2 reg

        # Sampling: Randomness helps stability
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),

        # Learning Rate: keep it low for precision
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.05),

        # Constants
        'n_estimators': 5000,
        'n_jobs': -1,
        'eval_metric': 'rmse',
        'early_stopping_rounds' :100,
    }

    # 3. Initialize Model

    # 4. Fit with Early Stopping and Pruning
    # The PruningCallback will stop unpromising trials early
    # pruning_callback = optuna.integration.XGBoostPruningCallback(trial, "validation_0-rmse")
    # model = xgb.XGBRegressor(**params,callbacks=[pruning_callback])
    model = xgb.XGBRegressor(**params)

    model.fit(
        X_trainer, y_trainer,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    preds = model.predict(X_val)
    rmseerror = rmse(y_val, preds)
    return rmseerror

In [ ]:
studyxgb = optuna.create_study(direction='minimize')
studyxgb.optimize(objectivexgb, n_trials=50)

[I 2026-01-24 14:54:54,233] A new study created in memory with name: no-name-96a71d29-e55d-4e49-adb5-dbaeeec1cad5
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:774: UserWarning: [14:55:01] WARNING: /workspace/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
[I 2026-01-24 14:55:01,605] Trial 0 finished with value: 0.0031212205738121754 and parameters: {'max_depth': 7, 'min_child_weight': 120, 'gamma': 0.41963387263066665, 'reg_alpha': 1.8866558411065508, 'reg_lambda': 3.7362420956936737, 'subsample': 0.6168787328929883, 'colsample_bytree': 0.8923934590642892, 'learning_rate': 0.021482741399390

In [ ]:
best_paramsxgb = studyxgb.best_params
# best_paramsxgb.update({
#     'n_estimators': 5000,          # Give it plenty of room
#     'objective': 'reg:squarederror',
#     'tree_method': 'hist',
#     'device': 'cuda',              # Use your GPU
#     'n_jobs': -1
# })

In [ ]:
final_modelxgb1 = xgb.XGBRegressor(**best_paramsxgb)

In [ ]:
final_modelxgb1.fit(
    X_trainer, y_trainer,
    eval_set=[(X_val, y_val)],   # Use your test/validation data here
    verbose=100                    # Print progress every 100 trees
)

# 5. Predict on the test data
final_predsxgb1 = final_modelxgb1.predict(X_test_sc)

[0]	validation_0-rmse:0.00312
[99]	validation_0-rmse:0.00312


In [ ]:
xgbrmse1 = rmse(y_test,final_predsxgb1)
xgbrmse1

0.003320483661080623

In [ ]:
best_paramsxgb.update({
    'n_estimators': 5000,          # Give it plenty of room
    'objective': 'reg:squarederror',
    'tree_method': 'hist',
    'device': 'cuda',              # Use your GPU
    'n_jobs': -1
})

In [ ]:
final_modelxgb2 = xgb.XGBRegressor(**best_paramsxgb)
final_modelxgb2.fit(
    X_trainer, y_trainer,
    eval_set=[(X_val, y_val)],   # Use your test/validation data here
    verbose=100                    # Print progress every 100 trees
)

# 5. Predict on the test data
final_predsxgb2 = final_modelxgb2.predict(X_test_sc)

[0]	validation_0-rmse:0.00312
[100]	validation_0-rmse:0.00312
[200]	validation_0-rmse:0.00312
[300]	validation_0-rmse:0.00312
[400]	validation_0-rmse:0.00312
[500]	validation_0-rmse:0.00312
[600]	validation_0-rmse:0.00312
[700]	validation_0-rmse:0.00312
[800]	validation_0-rmse:0.00312
[900]	validation_0-rmse:0.00312
[1000]	validation_0-rmse:0.00312
[1100]	validation_0-rmse:0.00312
[1200]	validation_0-rmse:0.00312
[1300]	validation_0-rmse:0.00312
[1400]	validation_0-rmse:0.00312
[1500]	validation_0-rmse:0.00312
[1600]	validation_0-rmse:0.00312
[1700]	validation_0-rmse:0.00312
[1800]	validation_0-rmse:0.00312
[1900]	validation_0-rmse:0.00312
[2000]	validation_0-rmse:0.00312
[2100]	validation_0-rmse:0.00312
[2200]	validation_0-rmse:0.00312
[2300]	validation_0-rmse:0.00312
[2400]	validation_0-rmse:0.00312
[2500]	validation_0-rmse:0.00312
[2600]	validation_0-rmse:0.00312
[2700]	validation_0-rmse:0.00312
[2800]	validation_0-rmse:0.00312
[2900]	validation_0-rmse:0.00312
[3000]	validation_0-rm

In [ ]:
xgbrmse2 = rmse(y_test,final_predsxgb2)
xgbrmse2

0.003320483098533976

In [ ]:
finalxgb = xgb.XGBRegressor(**best_paramsxgb)
finalxgb.fit(X_train_sc,y_train)
finalxgbpred = finalxgb.predict(X_test_sc)
rmse(y_test,finalxgbpred)

0.0033204827809437698

In [ ]:
import lightgbm as lgb

In [ ]:
def objective_lgbm(trial):
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'device': 'gpu',  # T4 GPU acceleration
        'gpu_platform_id': 0,
        'gpu_device_id': 0,
        'verbosity': -1,
        'n_estimators': 5000,

        # Key Architecture Params
        'boosting_type': trial.suggest_categorical('boosting_type', ['gbdt', 'goss']),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.05),

        # Noise Handling
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 20, 100),
        'lambda_l1': trial.suggest_float('lambda_l1', 0, 5),
        'lambda_l2': trial.suggest_float('lambda_l2', 0, 5),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 0.9),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 0.9)
    }

    # GOSS doesn't support bagging, so we fix it if GOSS is chosen
    if params['boosting_type'] == 'goss':
        params['bagging_fraction'] = 1.0
        params['bagging_freq'] = 0
    else:
        params['bagging_freq'] = 1

    model = lgb.LGBMRegressor(**params)

    model.fit(
        X_trainer, y_trainer,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(100, verbose=False)]
    )

    preds = model.predict(X_val)
    return rmse(y_val, preds)

In [ ]:
study_lgbm = optuna.create_study(direction='minimize')
study_lgbm.optimize(objective_lgbm, n_trials=30)

[I 2026-01-24 15:15:06,524] A new study created in memory with name: no-name-e252ae3f-32da-4923-8032-2199b4902ab3
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-01-24 15:16:18,294] Trial 0 finished with value: 0.00309109561984161 and parameters: {'boosting_type': 'goss', 'num_leaves': 57, 'max_depth': 10, 'learning_rate': 0.018883251202276713, 'min_data_in_leaf': 36, 'lambda_l1': 3.947241542744062, 'lambda_l2': 3.5288839742462534, 'feature_fraction': 0.6950224572361905, 'bagging_fraction': 0.7284073342340831}. Best is trial 0 with value: 0.00309109561984161.
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-01-24 15:17:12,690] Trial 1 finished with value: 0.00309409212288364 and parameters: {'b

In [ ]:
best_paramslgb = study_lgbm.best_params
finallgb = lgb.LGBMRegressor(**best_paramslgb)
finallgb.fit(X_train_sc,y_train)
finallgbpred = finallgb.predict(X_test_sc)
rmse(y_test,finallgbpred)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


0.0032883132435872096

In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 12.0 MB/s eta 0:00:00


In [ ]:
from catboost import CatBoostRegressor

In [ ]:
def objective_cat(trial):
    params = {
        'loss_function': 'RMSE',
        'eval_metric': 'RMSE',
        'task_type': 'GPU',
        'devices': '0',
        'iterations': 5000,
        'verbose': False,

        # Architecture
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.05),
        'depth': trial.suggest_int('depth', 4, 8), # CatBoost gets slow > 10
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),

        # The "Noise Killer" params
        'random_strength': trial.suggest_float('random_strength', 0.5, 2.0),
        'border_count': trial.suggest_int('border_count', 32, 255),
    }

    model = CatBoostRegressor(**params)

    model.fit(
        X_trainer, y_trainer,
        eval_set=(X_val, y_val),
        early_stopping_rounds=100
    )

    preds = model.predict(X_val)
    return rmse(y_val, preds)

In [ ]:
study_cat = optuna.create_study(direction='minimize')
study_cat.optimize(objective_cat, n_trials=15)

[I 2026-01-24 15:45:01,478] A new study created in memory with name: no-name-111c7759-55f6-4d78-8972-b7317f2f7344
[I 2026-01-24 15:45:27,330] Trial 0 finished with value: 0.003114100578046567 and parameters: {'learning_rate': 0.01486082121942078, 'depth': 6, 'l2_leaf_reg': 4.681116701720093, 'random_strength': 0.5147702844351796, 'border_count': 131}. Best is trial 0 with value: 0.003114100578046567.
[I 2026-01-24 15:45:53,632] Trial 1 finished with value: 0.0031101243985816943 and parameters: {'learning_rate': 0.012677650579353151, 'depth': 6, 'l2_leaf_reg': 9.338043264238024, 'random_strength': 1.752252319003933, 'border_count': 238}. Best is trial 1 with value: 0.0031101243985816943.
[I 2026-01-24 15:46:18,121] Trial 2 finished with value: 0.003113526075121722 and parameters: {'learning_rate': 0.013461915769284799, 'depth': 8, 'l2_leaf_reg': 4.08262067300306, 'random_strength': 1.1038733877185736, 'border_count': 55}. Best is trial 1 with value: 0.0031101243985816943.
[I 2026-01-24 

In [ ]:
best_paramscat = study_cat.best_params
finalcat = CatBoostRegressor(**best_paramscat)
finalcat.fit(X_train_sc,y_train)
finalcatpred = finalcat.predict(X_test_sc)
rmse(y_test,finalcatpred)

0:	learn: 0.0054799	total: 637ms	remaining: 10m 36s
1:	learn: 0.0054676	total: 1.23s	remaining: 10m 15s
2:	learn: 0.0054647	total: 1.89s	remaining: 10m 29s
3:	learn: 0.0054636	total: 2.64s	remaining: 10m 57s
4:	learn: 0.0054610	total: 3.14s	remaining: 10m 25s
5:	learn: 0.0054603	total: 3.74s	remaining: 10m 19s
6:	learn: 0.0054597	total: 4.34s	remaining: 10m 15s
7:	learn: 0.0054590	total: 4.86s	remaining: 10m 2s
8:	learn: 0.0054584	total: 5.51s	remaining: 10m 7s
9:	learn: 0.0054576	total: 6.06s	remaining: 9m 59s
10:	learn: 0.0054532	total: 6.63s	remaining: 9m 55s
11:	learn: 0.0054502	total: 7.14s	remaining: 9m 47s
12:	learn: 0.0054495	total: 7.71s	remaining: 9m 45s
13:	learn: 0.0054485	total: 8.23s	remaining: 9m 40s
14:	learn: 0.0054457	total: 8.55s	remaining: 9m 21s
15:	learn: 0.0054416	total: 8.84s	remaining: 9m 3s
16:	learn: 0.0054380	total: 9.12s	remaining: 8m 47s
17:	learn: 0.0054198	total: 9.46s	remaining: 8m 35s
18:	learn: 0.0054193	total: 9.75s	remaining: 8m 23s
19:	learn: 0.005

0.0034581538655604834

In [ ]:
rmse(y_test,finalcatpred)

0.0034581538655604834

In [ ]:
import joblib

In [ ]:
joblib.dump(finalcat,'finalcat.pkl')

['finalcat.pkl']

In [ ]:
joblib.dump(finallgb,'finallgb.pkl')

['finallgb.pkl']

In [ ]:
joblib.dump(finalxgb,'finalxgb.pkl')

['finalxgb.pkl']